# 03 · Volatility measure, targets and features

This notebook documents **Step 4**: how daily volatility is measured, what exactly the models will forecast, and which features they receive. The dataset is built by [`src/features/build_features.py`](../src/features/build_features.py) and stored in the PostgreSQL table `features`.

**Run first** (from the project root):
```bash
python -m src.data.setup_db              # adds the new table and column
python -m src.data.clean_prices          # two new cleaning rules for intraday prices
python -m src.features.build_features
```

**Timing convention.** A row dated *t* contains what is known **at the close of day t** (features) and what happens **afterwards** (targets):

```
            features: data up to t            targets: days t+1 ... t+h
  ... t-21 ........ t-4  t-3  t-2  t-1  [t] | t+1  t+2 ... t+5 ... t+22
                                 VIX: last US session before t
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

from src.db import get_engine
from src.features.build_features import FEATURE_COLUMNS, HORIZONS, TARGET_COLUMNS
from src.features.returns import load_clean_prices
from src.features.volatility import add_daily_variance, to_annual_vol
from src.viz import BLUE, GRID, INK, INK_2, MUTED, ORANGE, save, set_style

set_style()
engine = get_engine()
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:,.3f}".format)

## 1. Measuring volatility on a single day

Volatility on a given day is not observable. With one close per day, the only estimate is the squared return, which is unbiased but extremely noisy: a stock can swing 5% during the day and close unchanged. The intraday **high-low range** tells how far the price actually travelled.

The project's daily variance, `rv`, combines both sources of risk:

> `rv` = (overnight return)² + Garman-Klass variance of the trading session

When open/high/low are missing or were rejected by the cleaning rules, `rv` falls back to the squared close-to-close return (`rv_source = 'close'`).

In [ ]:
prices = load_clean_prices(engine)
rv = add_daily_variance(prices[prices["asset_type"] != "external"])

estimators = {"rv_close": "Squared return (close to close)", "rv_parkinson": "Parkinson (high-low, session only)",
              "rv_gk": "Garman-Klass (session only)", "rv": "rv = overnight + Garman-Klass"}

def lag1_autocorr(column):
    logs = np.log(rv[column].clip(lower=1e-6))
    return logs.groupby(rv["ticker"]).apply(lambda s: s.autocorr(1)).mean()

comparison = pd.DataFrame({
    "average level, stocks (annualised %)": [to_annual_vol(rv.loc[rv["asset_type"] == "stock", c].mean()) for c in estimators],
    "average level, indices (annualised %)": [to_annual_vol(rv.loc[rv["asset_type"] == "index", c].mean()) for c in estimators],
    "day-to-day autocorrelation": [lag1_autocorr(c) for c in estimators],
}, index=list(estimators.values()))
comparison

**Reading the table.**

* *Level:* the session-only estimators are lower than the close-to-close figure because they ignore the overnight gap. Adding it back, `rv` is close to the close-to-close level, as it should be: it measures the same risk, more precisely. It stays slightly below for indices, probably because an index's published high and low are smoothed (its constituents do not all trade at the same instant).
* *Autocorrelation:* a noisy measure looks random from one day to the next. The autocorrelation of the squared return is low because it is mostly noise; the range-based measures reveal how persistent volatility really is.

In [ ]:
def daily_vol(ticker, start, end):
    g = rv[(rv["ticker"] == ticker) & rv["date"].between(start, end)].set_index("date")
    return to_annual_vol(g["rv_close"]), to_annual_vol(g["rv"])

close_vol, range_vol = daily_vol("^FCHI", "2020-01-15", "2020-06-30")
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(close_vol.index, close_vol, color=MUTED, lw=1.2, label="From the squared return")
ax.plot(range_vol.index, range_vol, color=BLUE, lw=2, label="rv: overnight + intraday range")
ax.set_title("CAC 40 during the COVID-19 crash: two measures of daily volatility (annualised, %)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper right")
fig.tight_layout()
save(fig, "04_volatility_estimators")
plt.show()

The squared return drops to almost zero in the middle of the crash whenever the index happens to close near the previous close, although those were some of the most turbulent sessions on record. The range-based measure stays high. A model trained on the grey line would be learning from noise.

**How `rv` was obtained** (share of rows per year where the range was unavailable):

In [ ]:
source = rv.dropna(subset=["rv_source"]).assign(year=lambda d: d["date"].dt.year)
fallback = source.groupby("year")["rv_source"].apply(lambda s: (s == "close").mean())
print(f"Overall: {(source['rv_source'] == 'range').mean():.1%} of daily variances use the intraday range")
fallback[fallback > 0.002].map("{:.1%}".format).to_frame("close-to-close fallback").T

## 2. Targets

For each horizon *h* ∈ {1, 5, 22} trading days, the target is the **logarithm of the average daily variance over the next h days**. The logarithm is used because volatility is right-skewed (notebook 02); forecasts are converted back to volatility for evaluation.

In [ ]:
data = pd.read_sql("SELECT * FROM model_dataset", engine, parse_dates=["date"])
print(f"{len(data):,} rows | {data['ticker'].nunique()} series | {data['date'].min():%Y-%m-%d} to {data['date'].max():%Y-%m-%d}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharex=True)
for ax, h, label in zip(axes, HORIZONS, ["next day", "next week (5 days)", "next month (22 days)"]):
    vol = to_annual_vol(np.exp(data[f"target_{h}d"].dropna()))
    ax.hist(np.log10(vol), bins=80, color=BLUE)
    ax.set_title(f"{label}\nmedian {vol.median():.0f}%, skewness of log {np.log(vol).skew():.1f}", fontsize=10)
    ax.set_xticks(np.log10([5, 10, 20, 50, 100, 200]), ["5", "10", "20", "50", "100", "200"])
    ax.set_yticks([])
    ax.grid(visible=False)
    ax.set_xlabel("annualised volatility, % (log scale)")
fig.suptitle("Distribution of the three targets", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, "04_targets")
plt.show()

The one-day target is the most dispersed (one day of data is still a noisy measurement); averaging over a week or a month smooths it. Short horizons are therefore harder to forecast in relative terms, which Step 5 will quantify.

## 3. Features

All features are known at the close of day *t*.

| Group | Features | Motivation (notebook 02) |
|---|---|---|
| Own volatility | `log_rv_d`, `log_rv_w`, `log_rv_m`, `log_rv_q`, `log_rv_lt` | Clustering and long memory: day, week, month, quarter, long-run level (the first three are the HAR model) |
| Returns | `ret_d`, `ret_w`, `ret_m`, `neg_ret_d`, `down_share_m` | Leverage effect: falls raise volatility more than rises |
| Instability | `vol_of_vol_m` | How erratic volatility itself has been over the last month |
| Activity | `log_volume_ratio` | Volume relative to its monthly average (stocks only) |
| Market | `mkt_log_rv_d`, `mkt_log_rv_w`, `mkt_log_rv_m` | Co-movement: volatility of the median European stock |
| Implied volatility | `log_vix`, `vix_chg_5d` | Forward-looking expectations from the US options market, lagged one session |
| Calendar | `dow` | Day-of-week effects |

Static attributes (asset type, country, sector) are joined from `universe` through the view `model_dataset`.

In [ ]:
complete = data.dropna(subset=[c for c in FEATURE_COLUMNS if c != "log_volume_ratio"] + TARGET_COLUMNS)
corr = pd.DataFrame({f"{h} day" + ("s" if h > 1 else ""): complete[FEATURE_COLUMNS].corrwith(complete[f"target_{h}d"])
                     for h in HORIZONS})

diverging = LinearSegmentedColormap.from_list("orange_grey_blue", [ORANGE, "#ecebe7", BLUE])
fig, ax = plt.subplots(figsize=(5.6, 7))
im = ax.imshow(corr, cmap=diverging, vmin=-0.8, vmax=0.8, aspect="auto")
ax.set_xticks(range(3), corr.columns)
ax.set_yticks(range(len(corr)), corr.index)
for i in range(len(corr)):
    for j in range(3):
        v = corr.iloc[i, j]
        ax.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=9, color="white" if abs(v) > 0.5 else INK)
ax.set_title("Correlation of each feature with the target")
ax.set_xlabel("forecast horizon")
ax.grid(visible=False)
fig.colorbar(im, ax=ax, shrink=0.6, label="correlation")
fig.tight_layout()
save(fig, "04_feature_correlations")
plt.show()

**Reading the chart.** Past volatility dominates at every horizon, with the weekly and monthly averages more informative than the single last day (they are less noisy). The market-wide measures and the VIX come next. Return-based features have the expected negative sign (falls precede higher volatility). A simple correlation says nothing about what a feature adds *on top of* the others: that is measured by ablation in Step 9.

## 4. Safeguards against look-ahead

Using information from the future is the most damaging mistake in a forecasting project, because results look excellent and are worthless. Three safeguards:

1. **Unit tests** ([`tests/test_features.py`](../tests/test_features.py)): the features are rebuilt on data truncated at a date *T* and must be identical to the full-sample features for all dates ≤ *T*; changing today's prices must not change today's targets.
2. **VIX lag:** the feature uses the last US session strictly before *t* (checked below).
3. **Past-only windows:** every rolling statistic ends at *t*, and the long-run level is an expanding average.

In [ ]:
vix = prices[prices["ticker"] == "^VIX"].set_index("date")["close"]
sample = data[(data["ticker"] == "^FCHI") & data["date"].between("2020-03-09", "2020-03-16")][["date", "log_vix"]].copy()
sample["VIX used by the feature"] = np.exp(sample["log_vix"])
sample["VIX close on the same day"] = sample["date"].map(vix)
sample["VIX close on the previous US session"] = sample["date"].map(vix.shift(1))
sample.drop(columns="log_vix").set_index("date")

## 5. The modelling dataset

In [ ]:
summary = (data.assign(complete=data.index.isin(complete.index))
           .groupby("asset_type")
           .agg(series=("ticker", "nunique"), rows=("ticker", "size"), complete_rows=("complete", "sum"),
                first_date=("date", "min"), last_date=("date", "max")))
print(f"Rows with all features and targets: {len(complete):,} "
      f"({complete['date'].min():%Y-%m-%d} to {complete['date'].max():%Y-%m-%d})")
summary

In [ ]:
missing = data[FEATURE_COLUMNS + TARGET_COLUMNS].isna().mean().sort_values(ascending=False)
missing[missing > 0.0005].map("{:.1%}".format).to_frame("share of missing values").T

Missing values come from warm-up periods (a 66-day average needs 66 days; the long-run level needs a year), from the end of the sample (no future to build a target) and, for `log_volume_ratio`, from the indices, which have no volume.

**Sanity check.** A linear regression of each target on the three HAR features, fitted on all rows at once. This is an *in-sample* fit, not an evaluation (that requires the walk-forward framework of Step 5); it only confirms that the features carry signal, that the weight shifts from the last day to the monthly average as the horizon grows, and that the one-day target is the hardest to explain.

In [ ]:
rows = []
for h in HORIZONS:
    X = np.column_stack([np.ones(len(complete)), complete[["log_rv_d", "log_rv_w", "log_rv_m"]].to_numpy()])
    y = complete[f"target_{h}d"].to_numpy()
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    r2 = 1 - ((y - X @ beta) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    rows.append({"horizon (days)": h, "weight on day": beta[1], "weight on week": beta[2], "weight on month": beta[3], "R² (in-sample)": r2})
pd.DataFrame(rows).set_index("horizon (days)")

## Summary

* **Measure:** daily variance from the overnight gap and the intraday range (Garman-Klass), with a close-to-close fallback; far less noisy than squared returns.
* **Targets:** log of the average daily variance over the next 1, 5 and 22 trading days.
* **Features:** 18 variables in seven groups, each motivated by a stylised fact, all known at the close of day *t*.
* **Leakage:** enforced by unit tests on truncated data and by lagging the VIX.

Next: Step 5 builds the walk-forward evaluation framework and the first baselines.